In [ ]:
# PROCESO 1 - PROCESAMIENTO INICIAL SAR: SPLIT, CALIBRATION, DEBURST, TERRAIN CORRECTION,
# SPECKLE FILTER, LINEARTOFROMDB, CONVERT U8, ESCRITURA BEAM-DIMAP Y BANDA RENOMBRADA A "Sigma0".
# Las salidas (.dim + .data/Sigma0.img) de output_dir son la entrada de proceso2mosaico.ipynb.
# Al final imprime tiempos, estadisticas y caracteristicas del equipo, y guarda
# resumen_proceso1.json en output_dir.

import os
import re
import sys
import json
import time
import ctypes
import shutil
import platform as plataforma_sis
from esa_snappy import ProductIO, GPF, jpy

# CARPETA ENTRADA .ZIP DESCARGADOS
input_dir = r"C:\argenis\RADAR\MOSAICO\1_IMAGENES"
# CARPETA SALIDA BANDAS PROCESADAS
output_dir = r"C:\argenis\RADAR\MOSAICO\2_PROCES"
os.makedirs(output_dir, exist_ok=True)

# TIPOS JAVA NECESARIOS
HashMap = jpy.get_type('java.util.HashMap')
Integer = jpy.get_type('java.lang.Integer')
Double = jpy.get_type('java.lang.Double')

# ESTADISTICAS DE EJECUCION (solo miden y leen archivos; no cambian el procesamiento)
def fmt_tiempo(seg):
    """Formatea segundos como h:mm:ss"""
    seg = int(round(seg))
    return f"{seg // 3600}:{(seg % 3600) // 60:02d}:{seg % 60:02d}"

def info_salida(product_path):
    """Devuelve (MB en disco, columnas, filas) del producto BEAM-DIMAP.
    Si algo falla devuelve (None, None, None) sin afectar el procesamiento."""
    try:
        total = 0
        cols = rows = None
        if os.path.exists(product_path + ".dim"):
            total += os.path.getsize(product_path + ".dim")
        folder = product_path + ".data"
        if os.path.isdir(folder):
            for raiz, _, archivos in os.walk(folder):
                for a in archivos:
                    total += os.path.getsize(os.path.join(raiz, a))
            for a in os.listdir(folder):
                if a.endswith(".hdr"):
                    with open(os.path.join(folder, a), "r", encoding="utf-8", errors="ignore") as f:
                        txt = f.read()
                    m_c = re.search(r"samples\s*=\s*(\d+)", txt)
                    m_r = re.search(r"lines\s*=\s*(\d+)", txt)
                    if m_c and m_r:
                        cols, rows = int(m_c.group(1)), int(m_r.group(1))
                    break
        return total / 1024 ** 2, cols, rows
    except Exception:
        return None, None, None

# CARACTERISTICAS DEL EQUIPO (si algun dato no se puede leer, se muestra "-")
def ram_gb():
    """RAM total y disponible en GB."""
    try:
        import psutil
        vm = psutil.virtual_memory()
        return vm.total / 1024 ** 3, vm.available / 1024 ** 3
    except Exception:
        pass
    try:
        if sys.platform.startswith("win"):
            class MEMORYSTATUSEX(ctypes.Structure):
                _fields_ = [("dwLength", ctypes.c_ulong), ("dwMemoryLoad", ctypes.c_ulong),
                            ("ullTotalPhys", ctypes.c_ulonglong), ("ullAvailPhys", ctypes.c_ulonglong),
                            ("ullTotalPageFile", ctypes.c_ulonglong), ("ullAvailPageFile", ctypes.c_ulonglong),
                            ("ullTotalVirtual", ctypes.c_ulonglong), ("ullAvailVirtual", ctypes.c_ulonglong),
                            ("ullAvailExtendedVirtual", ctypes.c_ulonglong)]
            m = MEMORYSTATUSEX()
            m.dwLength = ctypes.sizeof(MEMORYSTATUSEX)
            ctypes.WinDLL("kernel32").GlobalMemoryStatusEx(ctypes.byref(m))
            return m.ullTotalPhys / 1024 ** 3, m.ullAvailPhys / 1024 ** 3
        with open("/proc/meminfo") as f:
            info = {l.split(":")[0]: int(l.split()[1]) for l in f if ":" in l}
        return info["MemTotal"] / 1024 ** 2, info.get("MemAvailable", 0) / 1024 ** 2
    except Exception:
        return None, None

def nombre_cpu():
    """Nombre comercial del procesador."""
    try:
        if sys.platform.startswith("win"):
            import winreg
            clave = winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE, r"HARDWARE\DESCRIPTION\System\CentralProcessor\0")
            return winreg.QueryValueEx(clave, "ProcessorNameString")[0].strip()
        with open("/proc/cpuinfo") as f:
            for l in f:
                if l.lower().startswith("model name"):
                    return l.split(":", 1)[1].strip()
    except Exception:
        pass
    return plataforma_sis.processor() or "-"

def memoria_pico_gb():
    """Memoria maxima usada por este proceso (Python + Java de SNAP), en GB."""
    try:
        import psutil
        mi = psutil.Process().memory_info()
        if hasattr(mi, "peak_wset"):
            return mi.peak_wset / 1024 ** 3
    except Exception:
        pass
    try:
        if sys.platform.startswith("win"):
            class PMC(ctypes.Structure):
                _fields_ = [("cb", ctypes.c_ulong), ("PageFaultCount", ctypes.c_ulong),
                            ("PeakWorkingSetSize", ctypes.c_size_t), ("WorkingSetSize", ctypes.c_size_t),
                            ("QuotaPeakPagedPoolUsage", ctypes.c_size_t), ("QuotaPagedPoolUsage", ctypes.c_size_t),
                            ("QuotaPeakNonPagedPoolUsage", ctypes.c_size_t), ("QuotaNonPagedPoolUsage", ctypes.c_size_t),
                            ("PagefileUsage", ctypes.c_size_t), ("PeakPagefileUsage", ctypes.c_size_t)]
            k32, psapi = ctypes.WinDLL("kernel32"), ctypes.WinDLL("psapi")
            k32.GetCurrentProcess.restype = ctypes.c_void_p
            psapi.GetProcessMemoryInfo.argtypes = [ctypes.c_void_p, ctypes.POINTER(PMC), ctypes.c_ulong]
            c = PMC()
            c.cb = ctypes.sizeof(PMC)
            if psapi.GetProcessMemoryInfo(k32.GetCurrentProcess(), ctypes.byref(c), c.cb):
                return c.PeakWorkingSetSize / 1024 ** 3
            return None
        import resource
        pico = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        return pico / 1024 ** 3 if sys.platform == "darwin" else pico / 1024 ** 2
    except Exception:
        return None

def info_equipo(carpeta):
    """Caracteristicas del equipo y de Java (SNAP) donde se procesa."""
    total, disponible = ram_gb()
    try:
        libre = shutil.disk_usage(carpeta).free / 1024 ** 3
    except Exception:
        libre = None
    try:
        rt = jpy.get_type('java.lang.Runtime').getRuntime()
        java_max, java_nucleos = rt.maxMemory() / 1024 ** 3, rt.availableProcessors()
    except Exception:
        java_max = java_nucleos = None
    return {
        "equipo": plataforma_sis.node(),
        "sistema": f"{plataforma_sis.system()} {plataforma_sis.release()} ({plataforma_sis.machine()})",
        "procesador": nombre_cpu(),
        "nucleos_logicos": os.cpu_count(),
        "ram_total_gb": round(total, 1) if total else None,
        "ram_disponible_inicio_gb": round(disponible, 1) if disponible else None,
        "disco_libre_salida_gb": round(libre, 1) if libre else None,
        "python": plataforma_sis.python_version(),
        "java_memoria_max_gb": round(java_max, 1) if java_max else None,
        "java_nucleos": java_nucleos,
    }

def imprimir_equipo(eq):
    print(" Equipo:", eq["equipo"], "|", eq["sistema"])
    print(" Procesador:", eq["procesador"], "|", eq["nucleos_logicos"], "nucleos logicos")
    print(f" RAM: total {eq['ram_total_gb'] or '-'} GB | disponible al inicio {eq['ram_disponible_inicio_gb'] or '-'} GB")
    print(f" Disco libre en la carpeta de salida: {eq['disco_libre_salida_gb'] or '-'} GB")
    print(f" Python {eq['python']} | memoria maxima de Java (SNAP): {eq['java_memoria_max_gb'] or '-'} GB"
          f" | nucleos que usa Java: {eq['java_nucleos'] or '-'}")

def process_subswath(prod, subswath, output_path):
    """Procesa cada subswath IWx y guarda el resultado"""
    print(f" Procesando {subswath} → {os.path.basename(output_path)}")
    
    #1 TOPSAR-SPLIT (todos los bursts por defecto)
    params = HashMap()
    params.put('subswath', subswath)
    split = GPF.createProduct('TOPSAR-Split', params, prod)
    
    #2 CALIBRATION
    params = HashMap()
    params.put('outputSigmaBand', True)
    cal = GPF.createProduct('Calibration', params, split)
    
    #3 TOPSAR-DEBURST
    deburst = GPF.createProduct('TOPSAR-Deburst', HashMap(), cal)
    
    #4 TERRAIN CORRECTION
    params = HashMap()
    params.put('demName', 'SRTM 3Sec')
    params.put('demResamplingMethod', 'BILINEAR_INTERPOLATION')
    params.put('imgResamplingMethod', 'BILINEAR_INTERPOLATION')
    params.put('pixelSpacingInMeter', Double(20.0))
    params.put('mapProjection', 'WGS84(DD)')
    params.put('maskOutAreaWithoutElevation', True)
    
    # MUESTRA POLARIZACIONES DISPONIBLES, ESCOGE VV
    band_names = list(deburst.getBandNames())
    print("Bandas detectadas:", band_names)
    vv_band = [b for b in band_names if "VV" in b]
    if vv_band:
        params.put('sourceBands', ','.join(vv_band))
        print("Banda seleccionada:", vv_band)
    else:
        params.put('sourceBands', ','.join(band_names))
        print("No se encontró VV, procesando todas")
    tc = GPF.createProduct('Terrain-Correction', params, deburst)
    
    #5 SPECKLE FILTER
    params = HashMap()
    params.put('filter', 'Boxcar')
    params.put('filterSizeX', Integer(3))
    params.put('filterSizeY', Integer(3))
    speckle = GPF.createProduct('Speckle-Filter', params, tc)
    
    #6 LINEAR TO FROM dB
    db = GPF.createProduct('LinearToFromdB', HashMap(), speckle)
    
    #7 CONVERT DATATYPE
    params = HashMap()
    params.put('targetDataType', 'uint8')
    params.put('scaling', 'linear')
    params.put('scalingRange', '95.0')
    convert = GPF.createProduct('Convert-Datatype', params, db)
    
    #8 WRITE
    ProductIO.writeProduct(convert, output_path, 'BEAM-DIMAP')
    print(f" {subswath} guardado")

def rename_bands_in_product(product_path, new_band_name):
    """Renombra las bandas en archivos .dim, .hdr, .img y XML de metadatos BEAM-DIMAP"""
    
    # BEAM-DIMAP crea una carpeta con sufijo .data y un archivo .dim
    folder_path = product_path + ".data"
    dim_file = product_path + ".dim"
    
    if not os.path.exists(folder_path):
        print(f"  Carpeta .data no encontrada: {folder_path}")
        return
    
    if not os.path.exists(dim_file):
        print(f"  Archivo .dim no encontrado: {dim_file}")
        return
    
    # Busca el archivo .hdr que empiece con "Sigma0_IW"
    hdr_file = None
    img_file = None
    old_band_name = None
    
    for filename in os.listdir(folder_path):
        if filename.endswith('.hdr') and 'Sigma0_IW' in filename:
            hdr_file = os.path.join(folder_path, filename)
            old_band_name = filename.replace('.hdr', '')
            img_file = os.path.join(folder_path, old_band_name + '.img')
            break
    
    if not hdr_file:
        print(f"  No se encontró archivo .hdr con Sigma0_IW en: {folder_path}")
        return
    
    print(f"  Encontrado banda antigua: {old_band_name}")
    
    try:
        # 1. ACTUALIZAR EL ARCHIVO .dim (XML principal)
        with open(dim_file, 'r', encoding='utf-8') as f:
            dim_content = f.read()
        
        new_dim_content = dim_content.replace(old_band_name, new_band_name)
        
        with open(dim_file, 'w', encoding='utf-8') as f:
            f.write(new_dim_content)
        print(f"  ✓ Archivo .dim actualizado")
        
        # 2. ACTUALIZAR EL ARCHIVO .hdr
        with open(hdr_file, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Reemplazar en el .hdr
        new_content = re.sub(
            r'band names = \{ ' + re.escape(old_band_name) + r' \}',
            f'band names = {{ {new_band_name} }}',
            content
        )
        
        with open(hdr_file, 'w', encoding='utf-8') as f:
            f.write(new_content)
        print(f"  ✓ Archivo .hdr actualizado")
        
        # 3. RENOMBRAR LOS ARCHIVOS .hdr E .img EN DISCO
        new_hdr_file = os.path.join(folder_path, new_band_name + '.hdr')
        new_img_file = os.path.join(folder_path, new_band_name + '.img')
        
        try:
            os.rename(hdr_file, new_hdr_file)
            print(f"  ✓ Archivo .hdr renombrado en disco: {new_band_name}.hdr")
        except Exception as e:
            print(f"  ✗ Error renombrando .hdr: {e}")
        
        if os.path.exists(img_file):
            try:
                os.rename(img_file, new_img_file)
                print(f"  ✓ Archivo .img renombrado en disco: {new_band_name}.img")
            except Exception as e:
                print(f"  ✗ Error renombrando .img: {e}")
        
        print(f"  ✓ Banda renombrada a: {new_band_name}")
            
    except Exception as e:
        print(f"  ✗ Error al procesar: {e}")

# REGISTRO DE TIEMPOS Y ESTADISTICAS
t_inicio_total = time.perf_counter()
fecha_inicio = time.strftime("%Y-%m-%d %H:%M:%S")
print("Inicio:", fecha_inicio)
equipo = info_equipo(output_dir)
imprimir_equipo(equipo)
registros = []          # una entrada por franja procesada
tiempos_archivo = {}    # segundos por archivo .zip
omitidos = []           # archivos que no se pudieron abrir

# BUCLE PARA TODOS LOS .ZIP
for fname in os.listdir(input_dir):
    if fname.lower().endswith(".zip"):
        zip_path = os.path.join(input_dir, fname)
        print(f"\n Procesando archivo: {fname}")
        t_inicio_archivo = time.perf_counter()
        prod = ProductIO.readProduct(zip_path)
        
        if prod is None:
            print("No se pudo abrir el producto, se omite.")
            omitidos.append(fname)
            continue
        
        print("Producto cargado")
        
        # NOMBRAR SALIDA en formato Sentinel-1 estándar
        parts = fname.replace(".zip", "").split("_")
        platform = parts[0]  # S1A
        pol = parts[4]  # 1SDV
        start_date = parts[5]  # 20220309T103420
        orbit = parts[7]  # 042243
        mission_id = parts[8]  # 0508EC
        unique_id = parts[9]  # 81AD
        
        # Extraer solo YYYYMMDD de la fecha
        date_only = start_date[:8]  # 20220309
        
        for iw in ["IW1", "IW2", "IW3"]:
            out_name = f"{platform}_{iw}_SLC__{pol}_{start_date}_{orbit}_{mission_id}_{unique_id}"
            out_path = os.path.join(output_dir, out_name)
            t_inicio_franja = time.perf_counter()
            process_subswath(prod, iw, out_path)
            
            # RENOMBRAR BANDA DESPUÉS DEL PROCESAMIENTO
            new_band_name = f"Sigma0"
            print(f" Renombrando banda a: {new_band_name}")
            rename_bands_in_product(out_path, new_band_name)
            
            # TIEMPO Y TAMAÑO DE LA FRANJA
            seg_franja = time.perf_counter() - t_inicio_franja
            mb, cols, rows = info_salida(out_path)
            registros.append({"archivo": fname, "franja": iw, "salida": out_name,
                              "segundos": seg_franja, "mb": mb, "cols": cols, "rows": rows})
            print(f" Tiempo {iw}: {fmt_tiempo(seg_franja)} ({seg_franja:.1f} s)")
        
        prod.dispose()
        tiempos_archivo[fname] = time.perf_counter() - t_inicio_archivo
        print(f" Tiempo del archivo: {fmt_tiempo(tiempos_archivo[fname])}")

print("\n Procesamiento finalizado. Archivos guardados en:", output_dir)

# RESUMEN DE TIEMPOS Y ESTADISTICAS
t_total = time.perf_counter() - t_inicio_total
print("\n" + "=" * 70)
print(" RESUMEN DE EJECUCION - PROCESO 1")
print("=" * 70)
print(" Fin:", time.strftime("%Y-%m-%d %H:%M:%S"))
print(f" Archivos .zip procesados: {len(tiempos_archivo)}")
print(f" Archivos omitidos (no se pudieron abrir): {len(omitidos)}")
for o in omitidos:
    print(f"   - {o}")
print(f" Franjas procesadas: {len(registros)}")
print(f" Tiempo total: {fmt_tiempo(t_total)} ({t_total / 60:.1f} min)")

if tiempos_archivo:
    seg_arch = list(tiempos_archivo.values())
    print(f" Tiempo promedio por archivo (3 franjas): {fmt_tiempo(sum(seg_arch) / len(seg_arch))}")

if registros:
    seg_fr = [r["segundos"] for r in registros]
    print(f" Tiempo por franja: promedio {fmt_tiempo(sum(seg_fr) / len(seg_fr))}"
          f" | minimo {fmt_tiempo(min(seg_fr))} | maximo {fmt_tiempo(max(seg_fr))}")
    for iw in ["IW1", "IW2", "IW3"]:
        seg_iw = [r["segundos"] for r in registros if r["franja"] == iw]
        if seg_iw:
            print(f"   {iw}: promedio {fmt_tiempo(sum(seg_iw) / len(seg_iw))} ({len(seg_iw)} franjas)")

    tamanos = [r["mb"] for r in registros if r["mb"] is not None]
    if tamanos:
        print(f" Tamaño en disco: total {sum(tamanos):.1f} MB"
              f" | promedio por franja {sum(tamanos) / len(tamanos):.1f} MB")

    print("\n Detalle por franja:")
    print(f" {'Franja':6s} {'Tiempo':>8s} {'MB':>8s} {'Columnas x filas':>18s}  Salida")
    for r in registros:
        mb_txt = f"{r['mb']:.1f}" if r["mb"] is not None else "-"
        dim_txt = f"{r['cols']} x {r['rows']}" if r["cols"] else "-"
        print(f" {r['franja']:6s} {fmt_tiempo(r['segundos']):>8s} {mb_txt:>8s} {dim_txt:>18s}  {r['salida']}")

# EQUIPO Y MEMORIA USADA
pico = memoria_pico_gb()
_, disponible_fin = ram_gb()
print("\n Equipo donde se proceso:")
imprimir_equipo(equipo)
print(f" Memoria maxima usada por el proceso (Python + SNAP): {pico:.1f} GB" if pico else
      " Memoria maxima usada por el proceso: -")
print(f" RAM disponible al final: {disponible_fin:.1f} GB" if disponible_fin else " RAM disponible al final: -")

# GUARDAR EL RESUMEN EN LA CARPETA DE SALIDA
try:
    resumen = {
        "inicio": fecha_inicio,
        "fin": time.strftime("%Y-%m-%d %H:%M:%S"),
        "tiempo_total_s": round(t_total, 1),
        "archivos_procesados": len(tiempos_archivo),
        "archivos_omitidos": omitidos,
        "franjas_procesadas": len(registros),
        "tiempo_por_archivo_s": {k: round(v, 1) for k, v in tiempos_archivo.items()},
        "franjas": [{**r, "segundos": round(r["segundos"], 1),
                     "mb": round(r["mb"], 1) if r["mb"] is not None else None} for r in registros],
        "equipo": equipo,
        "memoria_pico_gb": round(pico, 2) if pico else None,
        "ram_disponible_fin_gb": round(disponible_fin, 1) if disponible_fin else None,
    }
    ruta_resumen = os.path.join(output_dir, "resumen_proceso1.json")
    with open(ruta_resumen, "w", encoding="utf-8") as f:
        json.dump(resumen, f, indent=2, ensure_ascii=False)
    print(" Resumen guardado en:", ruta_resumen)
except Exception as e:
    print(" No se pudo guardar el resumen:", e)
print("=" * 70)


 Procesando archivo: S1A_IW_SLC__1SDV_20180823T101751_20180823T101818_023372_028AF1_7881.zip
Producto cargado
 Procesando IW1 → S1A_IW1_SLC__1SDV_20180823T101751_023372_028AF1_7881
Bandas detectadas: ['Sigma0_IW1_VH', 'Sigma0_IW1_VV']
Banda seleccionada: ['Sigma0_IW1_VV']
